# F01 â€” Build and execute CUDA in a T4 notebook

Run this notebook in Colab with a T4 runtime or Kaggle with a T4 accelerator and internet access. All compilation and GPU execution happen in this remote Linux session. Your local computer only needs a browser/editor and Git.

This notebook builds a native **vector smoke operation**, not native RMSNorm. It checks `y = 3 Ã— x`, then runs the PyTorch RMSNorm reference on CUDA tensors. No performance claim follows from a smoke pass. F02 will implement CUDA RMSNorm.

Each code cell explains its purpose. Run in order. If setup or a test fails, run the final export cell anyway and return the archive. A hard session loss may leave incomplete artifacts; it is never a pass.

## 1. Pin the source

A commit SHA identifies the exact source being tested. Copy the full 40-character commit from the feature PR, rather than testing a moving branch. A fresh directory prevents overwriting a previous attempt. This public checkout needs no GitHub token.

In [1]:
from pathlib import Path
import json
import os
import re
import shutil
import subprocess
import sys
import tempfile
import venv

REVISION = "a50b8ce6d88fb9c8419490a9fe984f2e2cf7cabf"
assert re.fullmatch(r"[0-9a-fA-F]{40}", REVISION), "Set REVISION to the PR's full commit SHA"
base = Path("/kaggle/working") if Path("/kaggle/working").is_dir() else Path("/content")
assert base.is_dir(), "Use a Colab or Kaggle Linux notebook"
workspace = Path(tempfile.mkdtemp(prefix="aegis-f01-", dir=base))
repo = workspace / "repo"
artifacts = workspace / "artifacts"
artifacts.mkdir()

def run(args, *, cwd=None, env=None):
    completed = subprocess.run(args, cwd=cwd, env=env or globals().get("project_env"), text=True, capture_output=True)
    with (artifacts / "setup-and-tests.log").open("a") as log:
        log.write("\n$ " + " ".join(map(str, args)) + "\n")
        log.write(completed.stdout + completed.stderr)
    print((completed.stdout + completed.stderr)[-4000:])
    completed.check_returncode()
    return completed

run(["git", "init", str(repo)])
run(["git", "remote", "add", "origin", "https://github.com/MutugiD/Aegis-Norm.git"], cwd=repo)
run(["git", "fetch", "--depth=1", "origin", REVISION], cwd=repo)
run(["git", "checkout", "--detach", "FETCH_HEAD"], cwd=repo)
actual = run(["git", "rev-parse", "HEAD"], cwd=repo).stdout.strip()
assert actual.lower() == REVISION.lower()
(artifacts / "commit.txt").write_text(actual + "\n")
print("Workspace:", workspace)

Initialized empty Git repository in /content/aegis-f01-72wt2h48/repo/.git/
hint: Using 'master' as the name for the initial branch. This default branch name
hint: is subject to change. To configure the initial branch name to use in all
hint: of your new repositories, which will suppress this warning, call:
hint: 
hint: 	git config --global init.defaultBranch <name>
hint: 
hint: Names commonly chosen instead of 'master' are 'main', 'trunk' and
hint: 'development'. The just-created branch can be renamed via this command:
hint: 
hint: 	git branch -m <name>


From https://github.com/MutugiD/Aegis-Norm
 * branch            a50b8ce6d88fb9c8419490a9fe984f2e2cf7cabf -> FETCH_HEAD

HEAD is now at a50b8ce /task: remove trailing blank line from package metadata

a50b8ce6d88fb9c8419490a9fe984f2e2cf7cabf

Workspace: /content/aegis-f01-72wt2h48


## 2. Inspect the notebook host

The **driver** lets the OS communicate with the GPU. The **CUDA toolkit** supplies `nvcc`, which compiles CUDA source on the VM's CPU. The **PyTorch wheel** supplies its own CUDA runtime dependencies; installing it does not install a complete compiler toolchain.

C2 uses PyTorch 2.14.0/cu126 and toolkit 12.6. This remains a qualification candidate. This cell stops before large downloads if the toolkit is missing or different. Do not force a driver/toolkit replacement just to bypass the check; export the report for compatibility review.

In [4]:
run(["nvidia-smi"])

# Keep the notebook's cu126 build target. Install toolkit 12.6
# alongside 12.8 without replacing the NVIDIA driver.
cuda_home = Path("/usr/local/cuda-12.6")
nvcc = cuda_home / "bin" / "nvcc"

if not nvcc.is_file():
    prefix = [] if os.geteuid() == 0 else ["sudo"]
    run(prefix + ["apt-get", "update"])
    run(prefix + ["apt-get", "install", "-y", "cuda-toolkit-12-6"])

assert nvcc.is_file(), "CUDA toolkit 12.6 installation did not provide nvcc"

os.environ["CUDA_HOME"] = str(cuda_home)
os.environ["CUDACXX"] = str(nvcc)
os.environ["PATH"] = str(cuda_home / "bin") + os.pathsep + os.environ["PATH"]

# Also update the subprocess environment if it already exists.
if "project_env" in globals():
    project_env["CUDA_HOME"] = str(cuda_home)
    project_env["CUDACXX"] = str(nvcc)
    project_env["PATH"] = str(cuda_home / "bin") + os.pathsep + project_env["PATH"]

toolkit = run([str(nvcc), "--version"]).stdout
assert re.search(r"release 12\.6\b", toolkit), "Expected the selected CUDA 12.6 compiler"
run(["c++", "--version"])

Sat Sep  5 20:36:54 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   50C    P8             14W /   70W |       0MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

CompletedProcess(args=['c++', '--version'], returncode=0, stdout='c++ (Ubuntu 11.4.0-1ubuntu1~22.04.3) 11.4.0\nCopyright (C) 2021 Free Software Foundation, Inc.\nThis is free software; see the source for copying conditions.  There is NO\nwarranty; not even for MERCHANTABILITY or FITNESS FOR A PARTICULAR PURPOSE.\n\n', stderr='')

## 3. Install in an isolated notebook environment

A Python virtual environment keeps project dependencies separate from provider-installed packages. Its CPU compiler and GPU are still those of the notebook VM. We explicitly choose the cu126 wheel, then install the package and its build/test dependencies. The package wheel includes native source files; the next step compiles them for this session.

This can take several minutes and requires disk space for PyTorch and its dependencies. Model weights are not downloaded in F01.

In [7]:
# Install venv/ensurepip support matching the notebook's Python version.
prefix = [] if os.geteuid() == 0 else ["sudo"]
venv_package = f"python{sys.version_info.major}.{sys.version_info.minor}-venv"

run(prefix + ["apt-get", "update"])
run(prefix + ["apt-get", "install", "-y", venv_package])

environment = workspace / "venv"

# Reuse the partial environment and capture any error in the existing log.
run([sys.executable, "-m", "venv", str(environment)])

Hit:1 https://cli.github.com/packages stable InRelease
Hit:2 https://cloud.r-project.org/bin/linux/ubuntu jammy-cran40/ InRelease
Hit:3 https://developer.download.nvidia.com/compute/cuda/repos/ubuntu2204/x86_64  InRelease
Hit:4 http://security.ubuntu.com/ubuntu jammy-security InRelease
Hit:5 https://r2u.stat.illinois.edu/ubuntu jammy InRelease
Hit:6 http://archive.ubuntu.com/ubuntu jammy InRelease
Hit:7 http://archive.ubuntu.com/ubuntu jammy-updates InRelease
Hit:8 https://ppa.launchpadcontent.net/deadsnakes/ppa/ubuntu jammy InRelease
Hit:9 http://archive.ubuntu.com/ubuntu jammy-backports InRelease
Hit:10 https://ppa.launchpadcontent.net/graphics-drivers/ppa/ubuntu jammy InRelease
Hit:11 https://ppa.launchpadcontent.net/ubuntugis/ppa/ubuntu jammy InRelease
Reading package lists...
W: Skipping acquire of configured file 'main/source/Sources' as repository 'https://r2u.stat.illinois.edu/ubuntu jammy InRelease' does not seem to provide it (sources.list entry misspelt?)

Reading package li

CompletedProcess(args=['/usr/bin/python3', '-m', 'venv', '/content/aegis-f01-72wt2h48/venv'], returncode=0, stdout='', stderr="Unable to symlink '/usr/bin/python3' to '/content/aegis-f01-72wt2h48/venv/bin/python3'\n")

## 4. Preflight and explicit native build

Preflight records the actual GPU, available memory, Python/PyTorch versions, compiler, toolkit and driver. `ready_for_build` means the build may be attempted; it is not a correctness result.

The smoke command invokes the C++/CUDA extension builder. Ninja coordinates compilation, `MAX_JOBS=2` limits host build parallelism, and architecture `7.5` targets T4. The binding receives a PyTorch tensor and passes its existing GPU pointer to the kernel; it does not copy the tensor through CPU memory.

The CUDA launcher uses PyTorch's current stream. Returning to Python does not mean the GPU is finished. The checks synchronize before comparing values, and exercise a non-default stream. Compiler output is saved to `build.log`; this cell may be quiet while compilation runs.

In [9]:
environment = workspace / "venv"
python = str(environment / "bin" / "python")
assert Path(python).is_file(), f"Environment interpreter missing: {python}"

project_env = os.environ.copy()
project_env["PATH"] = str(environment / "bin") + os.pathsep + project_env["PATH"]

run([python, "--version"])
run([python, "-m", "pip", "--version"])

run([python, "-m", "pip", "install", "torch==2.14.0",
     "--index-url", "https://download.pytorch.org/whl/cu126"])
run([python, "-m", "pip", "install",
     "-r", str(repo / "requirements-foundation.txt")])
run([python, "-m", "pip", "install", "--no-deps", str(repo)])
run([python, "-m", "pip", "check"])

run([python, "-m", "pip_audit", "--format", "json",
     "--output", str(artifacts / "installed-audit.json")])
run([python, "-m", "pip_audit",
     "-r", str(repo / "requirements-foundation.txt"),
     "--no-deps", "--disable-pip", "--format", "json",
     "--output", str(artifacts / "direct-audit.json")])

Python 3.13.15

pip 26.2.1 from /content/aegis-f01-72wt2h48/venv/lib/python3.13/site-packages/pip (python 3.13)

e,nvrtc,nvtx]==12.6.3; platform_system == "Linux"->torch==2.14.0)
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 216.6/216.6 MB 190.1 MB/s  0:00:01
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.6/23.6 MB 251.3 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 869.2/869.2 MB 16.9 MB/s  0:00:23
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 7.5/7.5 MB 89.0 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 246.9/246.9 MB 72.5 MB/s  0:00:03
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.1/2.1 MB 96.7 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.3/1.3 MB 35.8 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 6.3/6.3 MB 123.7 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 536.2/536.2 kB 27.7 MB/s  0:00:00


wnloading mdurl-0.1.2-py3-none-any.whl.metadata (1.6 kB)
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 818.2/818.2 kB 24.5 MB/s  0

CompletedProcess(args=['/content/aegis-f01-72wt2h48/venv/bin/python', '-m', 'pip_audit', '-r', '/content/aegis-f01-72wt2h48/repo/requirements-foundation.txt', '--no-deps', '--disable-pip', '--format', 'json', '--output', '/content/aegis-f01-72wt2h48/artifacts/direct-audit.json'], returncode=0, stdout='', stderr='WARNING:pip_audit._cli:--no-deps is supported, but users are encouraged to fully hash their pinned dependencies\nWARNING:pip_audit._cli:Consider using a tool like `pip-compile`: https://pip-tools.readthedocs.io/en/latest/#using-hashes\nNo known vulnerabilities found\n')

## 5. Run reference and native binding tests

The **reference** calculates RMSNorm with ordinary PyTorch operations. When its input tensor is on CUDA, those operations run on the T4. It accumulates in FP32 and casts normalized values before multiplying by the weight, preserving the specified rounding boundary.

`backend='cuda'` intentionally fails for RMSNorm in F01: the vector smoke kernel must never masquerade as native RMSNorm. `explain_dispatch` exposes that distinction. GPU tests require explicit opt-in so a CPU CI job cannot silently claim to have qualified CUDA.

In [10]:
test_env = project_env.copy()
test_env["AEGIS_RUN_GPU"] = "1"
run([python, "-m", "pytest", str(repo / "tests/test_reference.py"),
     str(repo / "tests/test_preflight.py"), str(repo / "tests/test_gpu_smoke.py"),
     "-v", "--junitxml=" + str(artifacts / "tests.xml")], cwd=workspace, env=test_env)

] PASSED [  5%]
repo/tests/test_reference.py::test_fp16_cast_boundary_changes_result PASSED [  7%]
repo/tests/test_reference.py::test_shape_empty_and_odd_width[shape0] PASSED [ 10%]
repo/tests/test_reference.py::test_shape_empty_and_odd_width[shape1] PASSED [ 13%]
repo/tests/test_reference.py::test_shape_empty_and_odd_width[shape2] PASSED [ 15%]
repo/tests/test_reference.py::test_shape_empty_and_odd_width[shape3] PASSED [ 18%]
repo/tests/test_reference.py::test_noncontiguous_mixed_dtype_and_gradients PASSED [ 21%]
repo/tests/test_reference.py::test_invalid_epsilon[0] PASSED             [ 23%]
repo/tests/test_reference.py::test_invalid_epsilon[-1] PASSED            [ 26%]
repo/tests/test_reference.py::test_invalid_epsilon[nan] PASSED           [ 28%]
repo/tests/test_reference.py::test_invalid_epsilon[inf] PASSED           [ 31%]
repo/tests/test_reference.py::test_invalid_epsilon[1e-50] PASSED         [ 34%]
repo/tests/test_reference.py::test_invalid_epsilon[1e+40] PASSED         [ 36%]


CompletedProcess(args=['/content/aegis-f01-72wt2h48/venv/bin/python', '-m', 'pytest', '/content/aegis-f01-72wt2h48/repo/tests/test_reference.py', '/content/aegis-f01-72wt2h48/repo/tests/test_preflight.py', '/content/aegis-f01-72wt2h48/repo/tests/test_gpu_smoke.py', '-v', '--junitxml=/content/aegis-f01-72wt2h48/artifacts/tests.xml'], returncode=0, stdout='============================= test session starts ==============================\nplatform linux -- Python 3.13.15, pytest-9.1.1, pluggy-1.6.0 -- /content/aegis-f01-72wt2h48/venv/bin/python\ncachedir: .pytest_cache\nrootdir: /content/aegis-f01-72wt2h48/repo\nconfigfile: pyproject.toml\ncollecting ... collected 38 items\n\nrepo/tests/test_reference.py::test_known_values_and_independent_output[dtype0] PASSED [  2%]\nrepo/tests/test_reference.py::test_known_values_and_independent_output[dtype1] PASSED [  5%]\nrepo/tests/test_reference.py::test_fp16_cast_boundary_changes_result PASSED [  7%]\nrepo/tests/test_reference.py::test_shape_empty_

## 6. Export evidence, including failures

Run this cell even if an earlier cell failed. The archive contains the commit, setup/test log, environment report and any smoke results. A successful smoke run additionally contains compiler output, case results, resolved package versions and SHA-256 file hashes.

Resolved versions are an environment snapshot, not proof that a different notebook will reproduce the run. Maintain the selected wheel index and native toolchain as recorded. F01 qualification remains pending until the run is reviewed. Full model/benchmark recovery workflows follow in F06.

In [11]:
archive = shutil.make_archive(str(workspace / "aegis-f01-evidence"), "zip", artifacts)
print("Evidence archive:", archive)
try:
    from google.colab import files
    files.download(archive)
except ImportError:
    from IPython.display import FileLink, display
    display(FileLink(archive))
    print("On Kaggle, also download the archive from the notebook output files.")

Evidence archive: /content/aegis-f01-72wt2h48/aegis-f01-evidence.zip


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>